# Lab 9 — Merge scans with transformation matrices

For each measurement we push a sensor-frame point $P_S = [d, 0, 0, 1]^T$ through
$$P_W = \;^W T_B \cdot \;^B T_S \cdot P_S$$

- $^B T_S$ orients the sensor on the body (per-reading rotation by yaw, plus mounting offset).
- $^W T_B$ places the body in the world at the robot's known tile position.

In [20]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----- Config -----
TILE_SIZE_M     = 0.3048   # 1 foot per tile
SENSOR_OFFSET_M = 0.08     # forward offset of ToF from rotation axis, in m
YAW_OFFSET_DEG  = -90.0      # global offset if every scan's start yaw isn't world +x
DATA_DIR        = Path('./Lab9Data/GoodData2')
WORLD_YAML      = DATA_DIR / 'world.yaml'   # reference map; None or missing file -> no overlay

# ----- Override filename-parsed poses (tile coordinates) -----
# Key: substring of the filename stem.  Value: (tile_x, tile_y)
POSE_OVERRIDES = {
    # 'C--3--2': (-3, -2),
}

# ----- Per-scan yaw correction, in degrees -----
YAW_OVERRIDES = {
     #'C--3--2': 180.0,
     #'C-5-3': 90
}

# ----- Colorblind-friendly palette (Okabe-Ito) + distinct markers -----
CB_COLORS  = ['#E69F00', '#56B4E9', '#009E73', '#D55E00',
              '#CC79A7', '#0072B2', '#000000', '#F0E442']
CB_MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', '*']

walls = [
    ((np.float64(-0.6812), np.float64(1.5347)), (np.float64(2.0476), np.float64(1.5139))),
    ((np.float64(2.0476), np.float64(1.5139)), (np.float64(1.9851), np.float64(-1.3139))),
    ((np.float64(1.9851), np.float64(-1.3139)), (np.float64(0.1729), np.float64(-1.2149))),
    ((np.float64(0.1729), np.float64(-1.2149)), (np.float64(0.1624), np.float64(-0.6213))),
    ((np.float64(0.1624), np.float64(-0.6213)), (np.float64(-0.3687), np.float64(-0.5952))),
    ((np.float64(-0.3687), np.float64(-0.5952)), (np.float64(-0.3896), np.float64(-1.4389))),
    ((np.float64(-0.3896), np.float64(-1.4389)), (np.float64(-1.8581), np.float64(-1.3868))),
    ((np.float64(-1.8529), np.float64(-1.3868)), (np.float64(-1.8269), np.float64(0.139))),
    ((np.float64(-1.8269), np.float64(0.139)), (np.float64(-0.7749), np.float64(0.1443))),
    ((np.float64(-0.7749), np.float64(0.1443)), (np.float64(-0.7281), np.float64(1.5295))),
    ((np.float64(0.6988), np.float64(-0.189)), (np.float64(1.4123), np.float64(-0.189))),
    ((np.float64(1.4383), np.float64(-0.2151)), (np.float64(1.4852), np.float64(0.6077))),
    ((np.float64(1.4748), np.float64(0.6234)), (np.float64(0.5478), np.float64(0.6442))),
    ((np.float64(0.5478), np.float64(0.6442)), (np.float64(0.6624), np.float64(-0.189))),
]


## Homogeneous transforms

In [21]:
def T_rot_z(theta_rad):
    c, s = np.cos(theta_rad), np.sin(theta_rad)
    T = np.eye(4)
    T[0,0], T[0,1] = c, -s
    T[1,0], T[1,1] = s,  c
    return T

def T_trans(dx, dy, dz=0.0):
    T = np.eye(4); T[0,3], T[1,3], T[2,3] = dx, dy, dz
    return T

def pose_to_T(x, y, theta_rad):
    """Body -> World transform for pose (x, y, theta)."""
    return T_trans(x, y) @ T_rot_z(theta_rad)

## Parse robot pose from filename, with overrides

In [22]:
def pose_from_filename(path):
    stem = Path(path).stem
    for key, val in POSE_OVERRIDES.items():
        if key in stem:
            return val
    m = re.search(r"C-(-?\d+)-(-?\d+)$", stem)
    if not m:
        raise ValueError(f"Couldn't parse pose from {stem}")
    return int(m.group(1)), int(m.group(2))

def yaw_offset_from_filename(path):
    """Look up per-scan yaw correction (deg); falls back to 0 if none set."""
    stem = Path(path).stem
    for key, val in YAW_OVERRIDES.items():
        if key in stem:
            return val
    return 0.0

## Transform one scan into the world frame

In [23]:
def transform_scan(df, tx_tiles, ty_tiles, yaw_offset_deg=0.0):
    x_r = tx_tiles * TILE_SIZE_M
    y_r = ty_tiles * TILE_SIZE_M
    # Global yaw offset goes into T_w_b (one-time rotation of the whole scan).
    T_w_b = pose_to_T(x_r, y_r, np.deg2rad(YAW_OFFSET_DEG))

    # Per-scan yaw correction is added to each reading's yaw directly —
    # same effect as rotating T_w_b further, but keeps the sensor->body
    # matrix parameterized by the corrected heading of each ray.
    yaw_rad = np.deg2rad(-df['yaw_deg'].values + yaw_offset_deg)
    d_m     = df['distance_mm'].values / 1000.0
    N = len(d_m)

    pts_world = np.empty((N, 2))
    for i in range(N):
        T_b_s = T_rot_z(yaw_rad[i]) @ T_trans(SENSOR_OFFSET_M, 0)
        P_s   = np.array([d_m[i], 0.0, 0.0, 1.0])
        P_w   = T_w_b @ T_b_s @ P_s
        pts_world[i] = P_w[:2]
    return pts_world, (x_r, y_r)

## Load the reference world map

Each entry in `world.yaml` under `world.lines` is stored as a string `(x, y), (x', y')`, so we pull the floats out directly with a regex instead of treating those values as YAML.

In [24]:
def load_world_lines(path):
    """Parse the "world.lines" entries in world.yaml into [((x1,y1),(x2,y2)), ...].
    Returns an empty list if the file is missing."""
    path = Path(path) if path else None
    if path is None or not path.exists():
        return []
    pat = re.compile(r"\(\s*(-?\d+\.?\d*)\s*,\s*(-?\d+\.?\d*)\s*\)"
                     r"\s*,\s*"
                     r"\(\s*(-?\d+\.?\d*)\s*,\s*(-?\d+\.?\d*)\s*\)")
    segments = []
    for line in path.read_text().splitlines():
        m = pat.search(line)
        if m:
            x1, y1, x2, y2 = map(float, m.groups())
            segments.append(((x1, y1), (x2, y2)))
    return segments

def draw_world(ax, segments, **kw):
    style = dict(color="black", linewidth=2.0, alpha=0.8, zorder=5)
    style.update(kw)
    for (p1, p2) in segments:
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], **style)

world_segments = load_world_lines(WORLD_YAML)
print(f"Loaded {len(world_segments)} reference wall segments from {WORLD_YAML}")

Loaded 13 reference wall segments from Lab9Data/GoodData2/world.yaml


## Load, merge, plot  —  with per-scan yaw sliders

Drag a slider to rotate that scan's rays about its scan origin. The plot redraws automatically; no need to re-run the cell. When you're happy with the values, click **Print YAW_OVERRIDES** and paste the output back into the config cell so the offsets persist.

In [32]:
import ipywidgets as widgets
from IPython.display import display

files = sorted(DATA_DIR.glob('mapping_data_*.csv'))

# Pre-load CSVs once so the slider callback only does the transform + draw.
scans = []
for i, f in enumerate(files):
    tx, ty = pose_from_filename(f)
    df     = pd.read_csv(f)
    scans.append({
        'file':   f,
        'key':    f.stem.split('_')[-1],           # e.g. 'C--3--2'
        'tx':     tx, 'ty': ty,
        'df':     df,
        'color':  CB_COLORS[i  % len(CB_COLORS)],
        'marker': CB_MARKERS[i % len(CB_MARKERS)],
    })

# One slider per scan, seeded with whatever YAW_OVERRIDES says.
sliders = {}
for i, s in enumerate(scans):
    sliders[f'scan_{i}'] = widgets.FloatSlider(
        value=yaw_offset_from_filename(s['file']),
        min=-180, max=180, step=1,
        description=f"({s['tx']},{s['ty']})",
        continuous_update=False,         # only redraw on mouse-up
        readout_format='.0f',
        layout=widgets.Layout(width='550px'),
        style={'description_width': '120px'},
    )

def redraw(**kwargs):
    fig, ax = plt.subplots(figsize=(9, 9))
    for i, s in enumerate(scans):
        yaw_corr = kwargs[f'scan_{i}']
        pts, origin = transform_scan(s['df'], s['tx'], s['ty'], yaw_corr)
        tag = f"({s['tx']},{s['ty']})"
        if yaw_corr:
            tag += f'  {yaw_corr:+.0f}°'
        ax.scatter(pts[:,0], pts[:,1], s=30,
                   color=s['color'], marker=s['marker'],
                   edgecolor='white', linewidth=0.5,
                   label=f"{tag} — {len(s['df'])} rays", zorder=3)
        ax.plot(*origin, marker='x', color=s['color'],
                markersize=12, mew=2.5, zorder=4)
    draw_world(ax, world_segments)
    ax.plot([], [], 'k-', linewidth=2, label='world.yaml')
    for i, (p1, p2) in enumerate(walls):
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'b-', linewidth=2, zorder=6,
                label='my map' if i == 0 else None)
    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    ax.set_title('Merged ToF map vs. reference world  (× = scan origin)')
    ax.legend(loc='best', fontsize=9)
    plt.tight_layout(); plt.show()

# Button: dump current slider values as a YAW_OVERRIDES dict you can paste back.
print_btn = widgets.Button(description='Print YAW_OVERRIDES', button_style='info')
print_out = widgets.Output()

def on_print(_):
    print_out.clear_output()
    with print_out:
        print('YAW_OVERRIDES = {')
        for i, s in enumerate(scans):
            v = sliders[f'scan_{i}'].value
            if v:
                print(f"    {s['key']!r}: {v:.1f},")
        print('}')
print_btn.on_click(on_print)

plot_out = widgets.interactive_output(redraw, sliders)
display(widgets.VBox(list(sliders.values())),
        widgets.HBox([print_btn]),
        print_out,
        plot_out)

Output()

Output()

## Draw your wall map

**How to use:**
1. Run this cell — a plot window opens
2. Click **two points** to define each wall segment; a line appears immediately
3. Keep clicking pairs until you've traced all the walls
4. When done, **close the plot window** (or press Enter in the cell)
5. The cell prints a ready-to-paste `walls` list and the simulator arrays

To redo a wall: re-run the cell — it starts fresh each time.
To keep walls from a previous run and add more: set `walls = <paste previous output>` in the config cell and the drawing session will start with those already drawn.

In [19]:
%matplotlib tk
# If 'tk' doesn't work on your system, try: %matplotlib qt  or  %matplotlib osx

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ── Rebuild the scatter from current slider/config state ──────────────────
files = sorted(DATA_DIR.glob('mapping_data_*.csv'))

fig, ax = plt.subplots(figsize=(10, 10))
fig.canvas.manager.set_window_title('Click pairs of points to draw walls — close when done')

for i, f in enumerate(files):
    color  = CB_COLORS[i % len(CB_COLORS)]
    marker = CB_MARKERS[i % len(CB_MARKERS)]
    tx, ty   = pose_from_filename(f)
    yaw_corr = yaw_offset_from_filename(f)
    df       = pd.read_csv(f)
    pts, _   = transform_scan(df, tx, ty, yaw_corr)
    ax.scatter(pts[:,0], pts[:,1], s=25, color=color, marker=marker,
               edgecolor='white', linewidth=0.4, alpha=0.75, zorder=3)

# Reference walls in grey
draw_world(ax, world_segments, color='grey', linewidth=1.5, alpha=0.5, zorder=4)

# Seed with any walls defined in the config / previous run
drawn_walls = list(walls)  # copy so we don't mutate the original
for (p1, p2) in drawn_walls:
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'b-', linewidth=2, zorder=5)

ax.set_aspect('equal'); ax.grid(alpha=0.25)
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
ax.set_title('Click pairs of points to draw walls.  Close window when done.')
plt.tight_layout()
plt.show(block=False)

print("Click point pairs on the plot.  Close the window when you're done.")
print("(If the plot isn't interactive, change the magic at the top of this cell.)")

while True:
    try:
        pts_clicked = fig.ginput(2, timeout=0)   # blocks until 2 clicks or window closes
    except Exception:
        break                                     # window was closed
    if len(pts_clicked) < 2:
        break                                     # window closed mid-click
    p1 = (round(pts_clicked[0][0], 4), round(pts_clicked[0][1], 4))
    p2 = (round(pts_clicked[1][0], 4), round(pts_clicked[1][1], 4))
    drawn_walls.append((p1, p2))
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'b-', linewidth=2, zorder=5)
    fig.canvas.draw()
    print(f"  added: {p1} → {p2}")

plt.close(fig)
%matplotlib inline

# ── Output ────────────────────────────────────────────────────────────────
print("\n# ── Paste this back into your config cell to persist ──")
print("walls = [")
for p1, p2 in drawn_walls:
    print(f"    ({p1}, {p2}),")
print("]")

print("\n# ── Simulator arrays ──")
x_starts = [p1[0] for p1, _ in drawn_walls]
y_starts = [p1[1] for p1, _ in drawn_walls]
x_ends   = [p2[0] for _, p2 in drawn_walls]
y_ends   = [p2[1] for _, p2 in drawn_walls]
print('x_starts =', x_starts)
print('y_starts =', y_starts)
print('x_ends   =', x_ends)
print('y_ends   =', y_ends)


Click point pairs on the plot.  Close the window when you're done.
(If the plot isn't interactive, change the magic at the top of this cell.)
  added: (np.float64(-0.6812), np.float64(1.5347)) → (np.float64(2.0476), np.float64(1.5139))
  added: (np.float64(2.0476), np.float64(1.5139)) → (np.float64(1.9851), np.float64(-1.3139))
  added: (np.float64(1.9851), np.float64(-1.3139)) → (np.float64(0.1729), np.float64(-1.2149))
  added: (np.float64(0.1729), np.float64(-1.2149)) → (np.float64(0.1624), np.float64(-0.6213))
  added: (np.float64(0.1624), np.float64(-0.6213)) → (np.float64(-0.3687), np.float64(-0.5952))
  added: (np.float64(-0.3687), np.float64(-0.5952)) → (np.float64(-0.3896), np.float64(-1.4389))
  added: (np.float64(-0.3896), np.float64(-1.4389)) → (np.float64(-1.8581), np.float64(-1.3868))
  added: (np.float64(-1.8529), np.float64(-1.3868)) → (np.float64(-1.8269), np.float64(0.139))
  added: (np.float64(-1.8269), np.float64(0.139)) → (np.float64(-0.7749), np.float64(0.1443))
 